# Paper 1 (Enhanced): Hybrid Deep Learning Approach for Skin Lesion Classification
### Fine-Tuning + Class Weighting + Ensemble (Hybrid Model)

**Goal:** Improve on the baseline 6-model comparison by building a hybrid ensemble that combines fine-tuned versions of the top 3 models.

**IMPORTANT: Run this in a FRESH Colab session (Runtime -> Restart session) with T4 GPU enabled.**

This notebook is self-contained -- it re-downloads the dataset and re-does the split, so it works even if your previous session disconnected.

## STEP 1: Kaggle Auth + Download Dataset

In [ ]:
import os
!mkdir -p ~/.kaggle
!echo "$KAGGLE_TOKEN" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!mkdir -p /content/ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d /content/ham10000
print("Dataset ready!")

## STEP 2: Imports

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, DenseNet121, MobileNetV2
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize

print("GPU Available:", tf.config.list_physical_devices('GPU'))

## STEP 3: Load Metadata + Same Split as Before (Reproducible with random_state=42)

In [ ]:
metadata = pd.read_csv('/content/ham10000/HAM10000_metadata.csv')
all_image_paths = glob.glob('/content/ham10000/**/*.jpg', recursive=True)
image_path_dict = {os.path.splitext(os.path.basename(p))[0]: p for p in all_image_paths}
metadata['image_path'] = metadata['image_id'].map(image_path_dict.get)
metadata = metadata.dropna(subset=['image_path']).reset_index(drop=True)

classes = sorted(metadata['dx'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
metadata['label_idx'] = metadata['dx'].map(class_to_idx)

# SAME random_state=42 as before -> same train/val/test split, so results are comparable
train_df, temp_df = train_test_split(metadata, test_size=0.3, stratify=metadata['dx'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['dx'], random_state=42)

print("Train:", train_df.shape, " Val:", val_df.shape, " Test:", test_df.shape)

## STEP 4: Compute Class Weights (Handles Imbalance)
This tells the model to pay more attention to rare classes like `df`, `vasc`, `akiec` instead of just predicting `nv` most of the time.

In [ ]:
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.array(range(len(classes))),
    y=train_df['label_idx'].values
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights (higher = rarer class, gets more attention):")
for i, c in enumerate(classes):
    print(f"  {c}: {class_weights[i]:.3f}")

## STEP 5: Data Generators

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def make_generators(preprocess_func):
    train_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_func,
        rotation_range=45, width_shift_range=0.1, height_shift_range=0.1,
        zoom_range=0.2, horizontal_flip=True, vertical_flip=True,
        brightness_range=[0.8, 1.2]
    )
    val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_func)

    train_gen = train_datagen.flow_from_dataframe(
        train_df, x_col='image_path', y_col='dx', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=classes, shuffle=True, seed=42
    )
    val_gen = val_test_datagen.flow_from_dataframe(
        val_df, x_col='image_path', y_col='dx', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=classes, shuffle=False
    )
    test_gen = val_test_datagen.flow_from_dataframe(
        test_df, x_col='image_path', y_col='dx', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=classes, shuffle=False
    )
    return train_gen, val_gen, test_gen

print("Ready.")

## STEP 6: Build + Fine-Tune Function (Two-Phase Training)
Phase 1: Train only the new head (like before).
Phase 2: Unfreeze the last ~30 layers of the base model and fine-tune with a very low learning rate. This is the key step that improves accuracy.

In [ ]:
def build_model(base_model_fn, num_classes=7):
    base = base_model_fn(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base.trainable = False

    inputs = tf.keras.Input(shape=(224,224,3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model, base

def train_with_finetuning(model_name, model_fn, preprocess_func, phase1_epochs=15, phase2_epochs=15):
    print(f"\n{'='*60}\n{model_name} - PHASE 1 (Frozen base, train head)\n{'='*60}")
    train_gen, val_gen, test_gen = make_generators(preprocess_func)
    model, base = build_model(model_fn)

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy', metrics=['accuracy'])

    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7)

    model.fit(train_gen, validation_data=val_gen, epochs=phase1_epochs,
              callbacks=[early_stop, reduce_lr], class_weight=class_weights, verbose=1)

    print(f"\n{'='*60}\n{model_name} - PHASE 2 (Fine-tuning last layers)\n{'='*60}")
    # Unfreeze the last 30 layers of the base model
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                  loss='categorical_crossentropy', metrics=['accuracy'])

    early_stop2 = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    reduce_lr2 = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-8)

    model.fit(train_gen, validation_data=val_gen, epochs=phase2_epochs,
              callbacks=[early_stop2, reduce_lr2], class_weight=class_weights, verbose=1)

    # Final evaluation
    test_gen.reset()
    y_pred_probs = model.predict(test_gen, verbose=0)
    y_true = test_gen.classes

    model.save(f'/content/{model_name}_finetuned.keras')
    return model, y_pred_probs, y_true

print("Function ready.")

## STEP 7: Fine-Tune the Top 3 Models (ResNet50, DenseNet121, MobileNetV2)

**This is the longest step: roughly 45-70 minutes PER model (2 phases x ~15-20 min), so budget 2.5-3.5 hours total.**

In [ ]:
TOP_MODELS = {
    'ResNet50': (ResNet50, resnet_preprocess),
    'DenseNet121': (DenseNet121, densenet_preprocess),
    'MobileNetV2': (MobileNetV2, mobilenet_preprocess)
}

finetuned_results = {}
y_true_ref = None

for name, (fn, prep) in TOP_MODELS.items():
    model, y_pred_probs, y_true = train_with_finetuning(name, fn, prep)
    finetuned_results[name] = y_pred_probs
    y_true_ref = y_true  # same for all since same test set

    y_pred = np.argmax(y_pred_probs, axis=1)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f"\n>>> {name} (fine-tuned) -> Accuracy: {acc:.4f}, F1: {f1:.4f}")

print("\n\nAll fine-tuning complete!")

## STEP 8: Build the Hybrid Ensemble (Average the 3 Fine-Tuned Models)
This combines the prediction probabilities from all 3 models by averaging them -- a standard and well-accepted "soft voting" ensemble technique.

In [ ]:
# Average the softmax probabilities from all 3 fine-tuned models
ensemble_probs = np.mean([finetuned_results['ResNet50'],
                           finetuned_results['DenseNet121'],
                           finetuned_results['MobileNetV2']], axis=0)

ensemble_pred = np.argmax(ensemble_probs, axis=1)

ensemble_acc = accuracy_score(y_true_ref, ensemble_pred)
ensemble_prec = precision_score(y_true_ref, ensemble_pred, average='macro', zero_division=0)
ensemble_rec = recall_score(y_true_ref, ensemble_pred, average='macro', zero_division=0)
ensemble_f1 = f1_score(y_true_ref, ensemble_pred, average='macro', zero_division=0)

y_true_bin = label_binarize(y_true_ref, classes=list(range(len(classes))))
ensemble_auc = roc_auc_score(y_true_bin, ensemble_probs, average='macro', multi_class='ovr')

print("HYBRID ENSEMBLE RESULTS:")
print(f"Accuracy:  {ensemble_acc:.4f}")
print(f"Precision: {ensemble_prec:.4f}")
print(f"Recall:    {ensemble_rec:.4f}")
print(f"F1-Score:  {ensemble_f1:.4f}")
print(f"AUC:       {ensemble_auc:.4f}")

## STEP 9: Before vs After Comparison Table

In [ ]:
final_comparison = pd.DataFrame({
    'Baseline ResNet50 (frozen)':      {'Accuracy': 0.7824, 'F1-Score': 0.5427, 'AUC': 0.9365},
    'Baseline DenseNet121 (frozen)':   {'Accuracy': 0.7745, 'F1-Score': 0.4794, 'AUC': 0.9310},
    'Baseline MobileNetV2 (frozen)':   {'Accuracy': 0.7638, 'F1-Score': 0.5166, 'AUC': 0.9182},
    'Hybrid Ensemble (fine-tuned + class-weighted + averaged)': {
        'Accuracy': ensemble_acc, 'F1-Score': ensemble_f1, 'AUC': ensemble_auc
    }
}).T

print(final_comparison.round(4))
final_comparison.to_csv('/content/hybrid_vs_baseline_results.csv')

improvement = (ensemble_acc - 0.7824) * 100
print(f"\nAccuracy improvement over best baseline model: {improvement:+.2f} percentage points")

## STEP 10: Confusion Matrix for Hybrid Ensemble

In [ ]:
cm = confusion_matrix(y_true_ref, ensemble_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix - Hybrid Ensemble Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('/content/hybrid_confusion_matrix.png', dpi=300)
plt.show()

print(classification_report(y_true_ref, ensemble_pred, target_names=classes))

## STEP 11: Download Results

In [ ]:
from google.colab import files
files.download('/content/hybrid_vs_baseline_results.csv')
files.download('/content/hybrid_confusion_matrix.png')

---
## What This Gives You
- A genuine "Hybrid Deep Learning" contribution: fine-tuning + class-weighting + ensemble
- A clear before/after accuracy comparison table
- This is the kind of contribution that justifies your paper's novelty over the base paper

**How to describe this in your paper:**
> "To address the limitations identified in prior work -- namely the use of frozen feature extractors and insufficient handling of class imbalance -- we propose a hybrid ensemble approach combining fine-tuned ResNet50, DenseNet121, and MobileNetV2 models with class-weighted loss. Prediction probabilities from the three fine-tuned models are averaged (soft voting) to produce the final classification."